In [1]:
from dash import Dash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64

from animal_shelter import AnimalShelter


############################
# Database Connection
############################

db = AnimalShelter()


def get_data(query=None):
    if query is None:
        query = {}

    data = db.read(query)

    if not data:
        return pd.DataFrame()

    df = pd.DataFrame.from_records(data)

    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    return df


def get_top_breeds(query=None, top_n=10):
    """
    Database enhancement:
    Uses MongoDB aggregation to count and rank breeds directly in the database.
    """
    if query is None:
        query = {}

    breed_data = db.aggregate_by_breed(query, top_n)

    if not breed_data:
        return pd.DataFrame(columns=["breed", "count"])

    return pd.DataFrame.from_records(breed_data)


def get_dashboard_summary(query=None):
    """
    Database enhancement:
    Uses MongoDB aggregation to calculate dashboard summary statistics.
    """
    if query is None:
        query = {}

    return db.aggregate_dashboard_summary(query)


def get_valid_map_row(dataframe, selected_rows):
    """
    Algorithm enhancement:
    Safely validates location data and selected row index before building the map.
    """
    required_columns = ["location_lat", "location_long"]

    if dataframe.empty:
        return None

    if not all(column in dataframe.columns for column in required_columns):
        return None

    valid_locations = dataframe.dropna(subset=required_columns)

    if valid_locations.empty:
        return None

    row_index = 0

    if selected_rows and len(selected_rows) > 0:
        row_index = selected_rows[0]

    if row_index >= len(valid_locations):
        row_index = 0

    return valid_locations.iloc[row_index]


def binary_search_breed(sorted_breeds, target_breed):
    """
    Algorithm enhancement:
    Performs binary search on a sorted breed list.
    """
    left = 0
    right = len(sorted_breeds) - 1
    target_breed = target_breed.lower().strip()

    while left <= right:
        middle = (left + right) // 2
        current_breed = sorted_breeds[middle].lower().strip()

        if current_breed == target_breed:
            return True
        elif current_breed < target_breed:
            left = middle + 1
        else:
            right = middle - 1

    return False


def get_breed_search_result(dataframe, search_value):
    """
    Algorithm enhancement:
    Uses binary search to determine whether a breed exists
    in the current filtered dataset.
    """
    if dataframe.empty or "breed" not in dataframe.columns or not search_value:
        return "Enter a breed name to search the current dataset."

    sorted_breeds = sorted(
        dataframe["breed"]
        .dropna()
        .unique()
    )

    found = binary_search_breed(sorted_breeds, search_value)

    if found:
        return f"Breed found: {search_value}"
    else:
        return f"Breed not found in current filtered dataset: {search_value}"


df = get_data()


############################
# Dashboard Setup
############################

app = Dash(__name__)

image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()


rescue_queries = {
    "reset": {},

    "water": {
        "animal_type": "Dog",
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        "$or": [
            {"breed": {"$regex": "Labrador", "$options": "i"}},
            {"breed": {"$regex": "Chesapeake", "$options": "i"}},
            {"breed": {"$regex": "Newfoundland", "$options": "i"}}
        ]
    },

    "mountain": {
        "animal_type": "Dog",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        "$or": [
            {"breed": {"$regex": "German Shepherd", "$options": "i"}},
            {"breed": {"$regex": "Alaskan Malamute", "$options": "i"}},
            {"breed": {"$regex": "Old English Sheepdog", "$options": "i"}},
            {"breed": {"$regex": "Siberian Husky", "$options": "i"}},
            {"breed": {"$regex": "Rottweiler", "$options": "i"}}
        ]
    },

    "disaster": {
        "animal_type": "Dog",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
        "$or": [
            {"breed": {"$regex": "Doberman", "$options": "i"}},
            {"breed": {"$regex": "German Shepherd", "$options": "i"}},
            {"breed": {"$regex": "Golden Retriever", "$options": "i"}},
            {"breed": {"$regex": "Bloodhound", "$options": "i"}},
            {"breed": {"$regex": "Rottweiler", "$options": "i"}}
        ]
    }
}


############################
# Layout
############################

app.layout = html.Div([

    html.Div([
        html.Img(
            src=f"data:image/png;base64,{encoded_image}",
            style={"height": "180px", "width": "auto"}
        ),

        html.H1("Grazioso Salvare Animal Rescue Dashboard"),

        html.H3("Enhanced Database Project"),

        html.P("Created by Ehab Abdelmeseh"),

        html.P(
            "Enhancement focus: MongoDB indexing, secure database connection handling, "
            "validated CRUD operations, aggregation queries, database-driven breed ranking, "
            "and improved dashboard reporting."
        )
    ], style={"textAlign": "center"}),

    html.Hr(),

    html.Label("Filter by Rescue Type:"),

    dcc.Dropdown(
        id="filter-type",
        options=[
            {"label": "Reset - Show All Animals", "value": "reset"},
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster or Individual Tracking", "value": "disaster"}
        ],
        value="reset",
        clearable=False,
        style={"width": "60%"}
    ),

    html.Br(),

    html.Label("Search Breed Using Binary Search:"),

    dcc.Input(
        id="breed-search",
        type="text",
        placeholder="Example: German Shepherd Mix",
        style={
            "width": "60%",
            "marginTop": "5px"
        }
    ),

    html.Div(
        id="breed-search-result",
        style={
            "fontWeight": "bold",
            "marginTop": "10px",
            "marginBottom": "20px"
        }
    ),

    html.Div(
        id="summary-id",
        style={
            "display": "flex",
            "gap": "20px",
            "justifyContent": "center",
            "marginBottom": "20px"
        }
    ),

    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": column, "id": column, "deletable": False, "selectable": True}
            for column in df.columns
        ],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "minWidth": "120px",
            "width": "120px",
            "maxWidth": "180px",
            "whiteSpace": "normal"
        },
        style_header={"fontWeight": "bold"}
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", style={"width": "50%"}),
            html.Div(id="map-id", style={"width": "50%"})
        ]
    )
])


############################
# Callbacks
############################

@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):
    query = rescue_queries.get(filter_type, {})
    filtered_df = get_data(query)
    return filtered_df.to_dict("records")


@app.callback(
    Output("summary-id", "children"),
    Input("filter-type", "value")
)
def update_summary(filter_type):
    query = rescue_queries.get(filter_type, {})
    summary = get_dashboard_summary(query)

    card_style = {
        "border": "1px solid #ccc",
        "padding": "15px",
        "width": "200px",
        "textAlign": "center",
        "borderRadius": "8px",
        "boxShadow": "1px 1px 4px #ccc"
    }

    return [
        html.Div([
            html.H4("Total Animals"),
            html.H2(summary["total_animals"])
        ], style=card_style),

        html.Div([
            html.H4("Unique Breeds"),
            html.H2(summary["unique_breeds"])
        ], style=card_style),

        html.Div([
            html.H4("Average Age Weeks"),
            html.H2(summary["average_age"])
        ], style=card_style)
    ]


@app.callback(
    Output("breed-search-result", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("breed-search", "value")
    ]
)
def update_breed_search(view_data, search_value):
    if not view_data:
        return "No data available to search."

    dff = pd.DataFrame.from_records(view_data)

    return get_breed_search_result(dff, search_value)


@app.callback(
    Output("graph-id", "children"),
    Input("filter-type", "value")
)
def update_graphs(filter_type):
    query = rescue_queries.get(filter_type, {})
    top_breeds = get_top_breeds(query, 10)

    if top_breeds.empty:
        return [html.P("No breed data available for chart.")]

    return [
        dcc.Graph(
            figure=px.bar(
                top_breeds,
                x="breed",
                y="count",
                title="Top 10 Breed Ranking Using MongoDB Aggregation",
                labels={"breed": "Breed", "count": "Number of Animals"}
            )
        )
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [
        {
            "if": {"column_id": column},
            "background_color": "#D2F3FF"
        }
        for column in selected_columns
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    if not view_data:
        return [html.P("No location data available.")]

    dff = pd.DataFrame.from_records(view_data)

    row = get_valid_map_row(dff, selected_rows)

    if row is None:
        return [html.P("No valid location data available.")]

    latitude = row["location_lat"]
    longitude = row["location_long"]

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(row.get("breed", "Unknown breed"))),
                        dl.Popup([
                            html.H3("Animal Information"),
                            html.P(f"Name: {row.get('name', 'Unknown')}"),
                            html.P(f"Breed: {row.get('breed', 'Unknown')}"),
                            html.P(f"Animal ID: {row.get('animal_id', 'Unknown')}")
                        ])
                    ]
                )
            ]
        )
    ]


############################
# Run Dashboard
############################

app.run(debug=False, port=8050)

MongoDB connection successful.


In [2]:
from animal_shelter import AnimalShelter

db = AnimalShelter()

print("Connected:", db.is_connected())
print("Total records:", db.count_documents())
print("Top breeds:", db.aggregate_by_breed(limit=5))
print("Summary:", db.aggregate_dashboard_summary())

MongoDB connection successful.
Connected: True
Total records: 10000
Top breeds: [{'count': 3009, 'breed': 'Domestic Shorthair Mix'}, {'count': 801, 'breed': 'Pit Bull Mix'}, {'count': 608, 'breed': 'Labrador Retriever Mix'}, {'count': 588, 'breed': 'Chihuahua Shorthair Mix'}, {'count': 324, 'breed': 'Domestic Medium Hair Mix'}]
Summary: {'total_animals': 10000, 'unique_breeds': 817, 'average_age': 116.1}


In [3]:
from animal_shelter import AnimalShelter

db = AnimalShelter()

print(db.health_check())
print("Total records:", db.count_documents())
print(db.aggregate_dashboard_summary())
print(db.aggregate_by_breed(limit=5))

MongoDB connection successful.
{'connected': True, 'database': 'aac', 'collection': 'animals', 'document_count': 10000}
Total records: 10000
{'total_animals': 10000, 'unique_breeds': 817, 'average_age': 116.1}
[{'count': 3009, 'breed': 'Domestic Shorthair Mix'}, {'count': 801, 'breed': 'Pit Bull Mix'}, {'count': 608, 'breed': 'Labrador Retriever Mix'}, {'count': 588, 'breed': 'Chihuahua Shorthair Mix'}, {'count': 324, 'breed': 'Domestic Medium Hair Mix'}]


In [4]:
from animal_shelter import AnimalShelter

db = AnimalShelter()

print(db.health_check())


MongoDB connection successful.
{'connected': True, 'database': 'aac', 'collection': 'animals', 'document_count': 10000}


In [5]:
results = db.text_search("Labrador")

print("Results:", len(results))

for animal in results[:5]:
    print(animal["breed"])

Results: 25
Labrador Retriever/Labrador Retriever
Labrador Retriever/Labrador Retriever
Labrador Retriever
Labrador Retriever
Labrador Retriever


In [6]:
record = db.read(limit=1)[0]

print(record)

{'_id': ObjectId('6a136281a4de23e1bc0cfa53'), 'rec_num': 2, 'age_upon_outcome': '1 year', 'animal_id': 'A725717', 'animal_type': 'Cat', 'breed': 'Domestic Shorthair Mix', 'color': 'Silver Tabby', 'date_of_birth': '5/2/2015', 'datetime': '5/6/2016 10:49', 'monthyear': '2016-05-06T10:49:00', 'name': '', 'outcome_subtype': 'SCRP', 'outcome_type': 'Transfer', 'sex_upon_outcome': 'Spayed Female', 'location_lat': 30.65259846, 'location_long': -97.74199635, 'age_upon_outcome_in_weeks': 52.92152778}


In [7]:
record = db.read(limit=1)[0]

db.update_by_id(
    str(record["_id"]),
    {"name": "CS499_Test"}
)

updated = db.read_one({"_id": record["_id"]})

print(updated["name"])

CS499_Test


In [8]:
db.update_by_id(
    str(record["_id"]),
    {"name": ""}
)

1